In [1]:
import yaml 
from pathlib import Path

import mlflow

from pytorch_pipeline import val
from pytorch_pipeline.train import build_pipeline_dataloaders, build_datasets, get_device
from pytorch_pipeline.utils import resolve_uri, Config, resolve_hardware_profile, get_current_git_branch, CLASS_ORDER
from pytorch_pipeline.utils.params import DatasetParams, DataLoadersParams, PathsParams
from pytorch_pipeline.review import review_label_issues


In [2]:
#Args
test = False
test_fraction = 0.05
seed = 42
model_name = 'cv_pheno_bioclip'
model_version = 1
config_path = Path("/home/etienne/projects/inat-phenology-cv/configs/local.yaml")

In [3]:
# Set up environment specific configs
with open(config_path, "r") as file:
    env_configs = yaml.safe_load(file)
paths_params = PathsParams(**env_configs["paths"])
dataloader_params = DataLoadersParams(**env_configs["dataloader_params"])
dataset_params = DatasetParams(
    **env_configs["dataset_params"], testing_frac=test_fraction)
hardware_profile = resolve_hardware_profile()
configs = Config(
    config_path,
    paths_params=paths_params,
    dataset_params=dataset_params,
    dataloaders_params=dataloader_params,
    hardware_profile=hardware_profile,
    git_branch=get_current_git_branch(),
    )
configs.test = test

In [4]:
# Load the model
mlflow.set_tracking_uri(resolve_uri())
model_uri = f"models:/{model_name}/{model_version}"
model = mlflow.pytorch.load_model(model_uri)

In [5]:
#Load model, dataset & dataloaders 
device = get_device()
datasets = build_datasets(configs, model, seed= seed)
_, val_loader, _ = build_pipeline_dataloaders(datasets, configs.dataloaders_params, seed=seed)

Running on cuda


In [6]:
#Run inference
obs_ids, raw_labels, raw_preds = val.execute(model=model, dataloader=val_loader, device=device, as_numpy=True)

In [7]:
from cleanlab.filter import find_label_issues
import numpy as np

means = []
issues = []

for i in range(3):
    # Format predicted probs for cleanlab
    labels = raw_labels[:,i].astype(int)
    pred_probs_pos = raw_preds[:,i]
    pred_probs_neg = 1 - pred_probs_pos
    pred_probs = np.column_stack((pred_probs_neg, pred_probs_pos))
    issue_mask = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    )
    means.append(issue_mask.mean())

    ordered_issue_indices = find_label_issues(
    labels=labels,
    pred_probs=pred_probs,
    return_indices_ranked_by="self_confidence"
    )
    
    issues.append([obs_ids[i] for i in ordered_issue_indices])

In [9]:
for i, _ in enumerate(CLASS_ORDER):
    print(f"{CLASS_ORDER[i]} {means[i]}")
    print(f" {len(issues[i])} issues")

Flowering 0.02023608768971332
 36 issues
Fruiting 0.07307476110174255
 130 issues
Flower_Budding 0.15177065767284992
 270 issues


In [27]:
class_idx = 2

x = review_label_issues(obs_ids= issues[class_idx],
 label_name= CLASS_ORDER[class_idx],
 db_path= "/home/etienne/projects/inat-phenology-cv/data/cv_raw.duckdb",
 image_dir= "/home/etienne/projects/inat-phenology-cv/data/images",
 table_name='cv_photos3',
 all_obs_ids=obs_ids,
 raw_labels=raw_labels,
 raw_preds=raw_preds,
 class_idx=class_idx
 )

In [ ]:
ids = []

for i in x:
    print(i['obs_id'])


261758594
270013526
116918680
265208040
307068898
261881558
54616082
67670642
15379803
117437090
282680326
270929149
198998681
142907323
48090251
167636758
120760429
140545116
82587983
156175126
67181750
134985475
226464721
125238936
308618644
84780993
269839911
228158642
208335598
127078045
89554115
187317929
44981518
129364710
136565558
93937132
217692464
102464205
93792078
27465048
232146397
103471850
172311191
54336729
185539490
256299909
83400158
37412234
77741992
82290076
240249685
334038443
149138952
304361491
312582105
148655996
205613366
233633609
57207143
73807314
28858086
267285630
71488641
127560184
237971491
232526243
268163105
326267939
303123230
144450420
118381453
128199055
284067608
120019645
61758667
157588961
295442971
304612753
123885285
121048586
125241181
30547411
28543476
75916020
102338203
211729106
18757365
87311834
194383072
120843136
173513667
59803857
310015534
133702935
102466202
13512655
149901090
216773248
187967397
97179140
31084055
97904776
163914681
23

: 